# Multiplication des pdf d'émission avec celle de la marée

In [ ]:
import fsspec
import s3fs
import numpy as np
import xdggs
import xarray as xr
import pandas as pd

In [ ]:
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

fs = s3fs.S3FileSystem(
    anon=storage_options["anon"],
    profile=storage_options["profile"],
    client_kwargs=storage_options["client_kwargs"],
)

tag_name = "A15789"
scratch_root = "s3://gfts-ifremer/eel/run/chue/test"
target_root = f"{scratch_root}/{tag_name}"

In [ ]:
fs.ls(target_root)

**Chargement de emission DST**

In [ ]:
emission = xr.open_dataset(f"{target_root}/emission.zarr", engine="zarr")  # .compute()
emission.longitude.compute()
emission.latitude.compute()
emission

**Chargement de emission tide**
On adapte l'échelle temporelle et on récupère la variable "tide found" avant le merge pour éviter de la multiplier lors du combine. 

In [ ]:
emission_tide = xr.open_dataset("A15789_tide_pdf_healpix.zarr", engine="zarr")
emission_tide["pdf_tide"] = emission_tide["pdf"]

origin = pd.to_datetime(emission["time"].isel(time=0).values)
delta = pd.to_timedelta(emission_tide.time.values, unit="h")
emission_tide = emission_tide.assign_coords(time=origin + delta)

tide_found = emission_tide["tide_found"]
emission_tide = emission_tide.drop_vars("tide_found")
emission_tide

## On sélectionne les cellules communes 
Les deux pdf n'ont pas les mêmes bbox de base, il faut donc sélectionner la bbox commune. 

In [ ]:
common_cells = np.intersect1d(emission["cell_ids"], emission_tide["cell_ids"])
common_cells.shape

On crée les deux sous datasets que l'on va merge pour éviter les effets de bords. 

*Il y a peut être un moyen d'efectuer les changement directement sur les datasets originels pour éviter de prendre trop de mémoire.*

In [ ]:
emission_small = emission.set_index(cells="cell_ids").sel(cells=common_cells)
emission_tide_small = emission_tide.set_index(cells="cell_ids").sel(cells=common_cells)
emission_small["dst_pdf"] = emission_small["pdf"]
emission_small

## On merge les deux datasets puis on combine les deux pdf 

On ajoute aussi la variable tide_found pour le multi-sigmas

Attention lorsqu'on utilise normalize pdf pour faire le combine, on multiplie toute les variables entre elles dont **tide_found**, on obtient donc une carte de 0 qui est ensuite transformée en carte de nan dans normalize_pdf.

**Bien rajouté tide_found après le normalize/combine**

In [ ]:
from pangeo_fish.helpers import normalize_pdf

emission_with_tide = emission_tide_small.drop_vars("pdf_tide").merge(
    emission_small.drop_vars("pdf")
)
default_chunk_dims = {"time": 24}

In [ ]:
combined_diff_bathy = normalize_pdf(
    ds=emission_with_tide,
    chunks=default_chunk_dims,
    dims=["cells"],
)[0]
combined_diff_bathy
combined_diff_bathy = combined_diff_bathy.assign_coords(
    cell_ids=("cells", combined_diff_bathy["cells"].values)
)  # use cell_ids instead of cells
combined_diff_bathy["tide_found"] = tide_found  # add of tide_found variable
combined_diff_bathy

# emission_with_tide = emission_tide_small.merge(emission_small.drop_vars('pdf'))
# emission_with_tide = emission_with_tide.assign_coords(cell_ids=("cells", emission_with_tide["cells"].values))

**Plot of the combine pdf**

In [ ]:
combined_diff_bathy.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)

Multiple visualisations 

In [ ]:
emission_with_tide.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8) | emission_with_tide.pdf_tide.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(
    alpha=0.8
) | emission_with_tide.pdf_final.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(
    alpha=0.8
)

### Saving of the combined dataset

In [ ]:
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}

combined_diff_bathy.compute().to_zarr(
    f"{target_root}/emission_w_tide_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_version=2,
    storage_options=storage_options,
)

In [ ]:
combined_diff_bathy = xr.open_dataset(
    f"{target_root}/emission_w_tide_{tag_name}.zarr", engine="zarr"
)
combined_diff_bathy

# Changement du sigma 

In [ ]:
def sigma_to_speed_kmh(
    sigma, earth_radius_km=6371, truncate=4, adjustment_factor=5, timedelta_h=1
):
    if sigma in (None, "None", np.nan):
        return np.nan
    try:
        sigma = float(sigma)
    except (ValueError, TypeError):
        return np.nan
    return sigma * (earth_radius_km / (timedelta_h))


def speed_kmh_to_sigma(speed, earth_radius_km=6371, timedelta_h=1):
    if speed in (None, "None", np.nan):
        return np.nan
    try:
        speed = float(speed)
    except (ValueError, TypeError):
        return np.nan
    return speed / (earth_radius_km / (timedelta_h))

In [ ]:
sigma_slow = speed_kmh_to_sigma(30 / 24)
sigma_fast = speed_kmh_to_sigma(100 / 24)
print("sigma_slow", sigma_slow)
print("sigma_fast", sigma_fast)

# Test bathy_pdf

In [ ]:
emission_bathy = xr.open_dataset(
    f"{target_root}/emission_w_bathy_pdf_A15789.zarr", engine="zarr"
)
emission_bathy

In [ ]:
emission_bathy.pdf.compute().dggs.decode(
    {"grid_name": "healpix", "level": 12, "indexing_scheme": "nested"}
).dggs.explore(alpha=0.8)